# Phase 3A — Figure 4: Log10 Absolute EEG Band Power

Secondary metric: **log10 absolute ROI band power**. This notebook generates PRE/POST plots and participant-level POST−PRE change plots for each task-condition. Visualization only; no new inferential tests. Outlier flags do not automatically exclude records.

In [1]:
import os, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import t
from pathlib import Path

candidates = [Path.cwd()/"eeg_roi_power_long.csv", Path.home()/"Term Project"/"analysis"/"eeg_roi_power_long.csv", Path.home()/"eeg_roi_power_long.csv", Path("/mnt/data/eeg_roi_power_long.csv")]
csv_path = next((p for p in candidates if p.exists()), None)
if csv_path is None:
    matches = glob.glob(str(Path.home()/"**/eeg_roi_power_long.csv"), recursive=True)[:50]
    csv_path = Path(matches[0]) if matches else None
if csv_path is None: raise FileNotFoundError("Could not locate eeg_roi_power_long.csv. Put it beside this notebook or in Term Project/analysis.")
df = pd.read_csv(csv_path)
required = {'subject_id','session','condition','band','roi','absolute_power_roi','task','group'}
missing = required - set(df.columns)
if missing: raise ValueError(f"Missing required columns: {missing}")
if (df['absolute_power_roi'] <= 0).any(): raise ValueError("Non-positive absolute power found; log10 is undefined.")
df['log10_absolute_power'] = np.log10(df['absolute_power_roi'])
output_dir = csv_path.parent / 'phase3_figures' / 'figure4_log10_absolute_power'
output_dir.mkdir(parents=True, exist_ok=True)
print(f'Input: {csv_path} | rows={len(df)} | output={output_dir}')
print('Tasks:', sorted(df.task.unique()), '| conditions:', sorted(df.condition.unique()))

Input: /Users/ananyosen/Documents/Term Project/analysis/eeg_roi_power_long.csv | rows=8000 | output=/Users/ananyosen/Documents/Term Project/analysis/phase3_figures/figure4_log10_absolute_power
Tasks: ['Math', 'Reading'] | conditions: ['Activity', 'Activity1', 'Activity2', 'Baseline']


## QC and paired dataset

In [2]:
keys = ['subject_id','task','condition','band','roi','session']
if df.duplicated(keys).any(): raise ValueError('Duplicate participant/task/condition/band/ROI/session records')
if df['log10_absolute_power'].isna().any(): raise ValueError('Missing log10 absolute values')
wide = df.pivot(index=['subject_id','task','condition','band','roi','group'], columns='session', values='log10_absolute_power').reset_index()
paired = wide.dropna(subset=['PRE','POST']).copy()
paired['change_POST_minus_PRE'] = paired['POST'] - paired['PRE']
print(f"Records: {len(df):,}; matched PRE-POST rows: {len(paired):,}; unpaired rows: {len(wide)-len(paired):,}")
print(paired.groupby(['task','condition']).size().to_string())

Records: 8,000; matched PRE-POST rows: 4,000; unpaired rows: 0
task     condition
Math     Activity1    850
         Activity2    850
         Baseline     850
Reading  Activity     725
         Baseline     725


## Figure 4A — PRE vs POST log10 absolute power

In [3]:
bands = ['delta','theta','alpha','beta','gamma']
rois = ['Frontal','Central','Temporal','Parietal','Occipital']
conditions_by_task = {'Math':['Baseline','Activity1','Activity2'], 'Reading':['Baseline','Activity']}
summary_prepost = df.groupby(['task','condition','band','roi','group','session'], observed=True)['log10_absolute_power'].agg(['mean','std','count']).reset_index()
summary_prepost['sem'] = summary_prepost['std'] / np.sqrt(summary_prepost['count'])
summary_prepost['ci95'] = summary_prepost.apply(lambda r: t.ppf(.975, r['count']-1)*r['sem'] if r['count']>1 else np.nan, axis=1)
fig_paths=[]
for task, conditions in conditions_by_task.items():
    for cond in conditions:
        fig, axes = plt.subplots(5,5, figsize=(19,15), sharex=True)
        for i, band in enumerate(bands):
            for j, roi in enumerate(rois):
                ax=axes[i,j]
                sub=summary_prepost[(summary_prepost.task==task)&(summary_prepost.condition==cond)&(summary_prepost.band==band)&(summary_prepost.roi==roi)]
                for group, color in [('Control','#4C9BD6'),('Experiment','#F28E2B')]:
                    g=sub[sub.group==group]
                    means=[]; cis=[]
                    for sess in ['PRE','POST']:
                        r=g[g.session==sess]
                        means.append(float(r['mean'].iloc[0]) if len(r) else np.nan)
                        cis.append(float(r['ci95'].iloc[0]) if len(r) and pd.notna(r['ci95'].iloc[0]) else np.nan)
                    ax.errorbar([0,1],means,yerr=cis,marker='o',capsize=3,label=group,color=color)
                if i==0: ax.set_title(roi)
                if j==0: ax.set_ylabel(f'{band.title()}\nlog10 absolute power')
                ax.set_xticks([0,1],['PRE','POST']); ax.grid(alpha=.25)
        handles,labels=axes[0,0].get_legend_handles_labels(); fig.legend(handles,labels,loc='upper right')
        fig.suptitle(f'{task} — {cond}: Log10 Absolute Power PRE–POST',y=1.01,fontsize=16)
        fig.tight_layout()
        path=output_dir/f'figure4_{task.lower()}_{cond.lower()}_log10_absolute_pre_post.png'
        fig.savefig(path,dpi=300,bbox_inches='tight'); plt.close(fig); fig_paths.append(path)
print('PRE/POST figures:',len(fig_paths))
for p in fig_paths: print(p)

PRE/POST figures: 5
/Users/ananyosen/Documents/Term Project/analysis/phase3_figures/figure4_log10_absolute_power/figure4_math_baseline_log10_absolute_pre_post.png
/Users/ananyosen/Documents/Term Project/analysis/phase3_figures/figure4_log10_absolute_power/figure4_math_activity1_log10_absolute_pre_post.png
/Users/ananyosen/Documents/Term Project/analysis/phase3_figures/figure4_log10_absolute_power/figure4_math_activity2_log10_absolute_pre_post.png
/Users/ananyosen/Documents/Term Project/analysis/phase3_figures/figure4_log10_absolute_power/figure4_reading_baseline_log10_absolute_pre_post.png
/Users/ananyosen/Documents/Term Project/analysis/phase3_figures/figure4_log10_absolute_power/figure4_reading_activity_log10_absolute_pre_post.png


## Figure 4B — Participant-level POST−PRE change

In [4]:
change_summary = paired.groupby(['task','condition','band','roi','group'], observed=True)['change_POST_minus_PRE'].agg(['mean','std','count']).reset_index()
change_summary['sem']=change_summary['std']/np.sqrt(change_summary['count'])
change_summary['ci95']=change_summary.apply(lambda r: t.ppf(.975,r['count']-1)*r['sem'] if r['count']>1 else np.nan,axis=1)
change_paths=[]
for task, conditions in conditions_by_task.items():
    for cond in conditions:
        fig,axes=plt.subplots(5,5,figsize=(19,15))
        for i,band in enumerate(bands):
            for j,roi in enumerate(rois):
                ax=axes[i,j]; sub=paired[(paired.task==task)& (paired.condition==cond)& (paired.band==band)& (paired.roi==roi)]
                for x,group,color in [(0,'Control','#4C9BD6'),(1,'Experiment','#F28E2B')]:
                    g=sub[sub.group==group]
                    if len(g):
                        ax.scatter(np.full(len(g),x)+np.random.default_rng(12).normal(0,.025,len(g)),g.change_POST_minus_PRE,s=16,alpha=.55,color=color)
                ss=change_summary[(change_summary.task==task)& (change_summary.condition==cond)& (change_summary.band==band)& (change_summary.roi==roi)]
                for x,group,color in [(0,'Control','#4C9BD6'),(1,'Experiment','#F28E2B')]:
                    r=ss[ss.group==group]
                    if len(r): ax.errorbar(x,r['mean'].iloc[0],yerr=r['ci95'].iloc[0],fmt='o',capsize=3,color=color)
                ax.axhline(0,linestyle='--',linewidth=.8,color='gray')
                if i==0: ax.set_title(roi)
                if j==0: ax.set_ylabel(f'{band.title()}\nΔ log10 power')
                ax.set_xticks([0,1],['Control','Experiment']); ax.grid(alpha=.2)
        fig.suptitle(f'{task} — {cond}: Log10 Absolute Power POST−PRE Change',y=1.01,fontsize=16)
        fig.tight_layout()
        path=output_dir/f'figure4_{task.lower()}_{cond.lower()}_log10_absolute_change.png'
        fig.savefig(path,dpi=300,bbox_inches='tight'); plt.close(fig); change_paths.append(path)
print('Change figures:',len(change_paths))
for p in change_paths: print(p)

Change figures: 5
/Users/ananyosen/Documents/Term Project/analysis/phase3_figures/figure4_log10_absolute_power/figure4_math_baseline_log10_absolute_change.png
/Users/ananyosen/Documents/Term Project/analysis/phase3_figures/figure4_log10_absolute_power/figure4_math_activity1_log10_absolute_change.png
/Users/ananyosen/Documents/Term Project/analysis/phase3_figures/figure4_log10_absolute_power/figure4_math_activity2_log10_absolute_change.png
/Users/ananyosen/Documents/Term Project/analysis/phase3_figures/figure4_log10_absolute_power/figure4_reading_baseline_log10_absolute_change.png
/Users/ananyosen/Documents/Term Project/analysis/phase3_figures/figure4_log10_absolute_power/figure4_reading_activity_log10_absolute_change.png


## Export summaries and manifest

In [5]:
summary_prepost.to_csv(output_dir/'figure4_log10_absolute_prepost_summary.csv',index=False)
change_summary.to_csv(output_dir/'figure4_log10_absolute_change_summary.csv',index=False)
manifest=pd.DataFrame({'figure_file':[str(p) for p in fig_paths+change_paths]})
manifest.to_csv(output_dir/'figure4_output_manifest.csv',index=False)
print('Exported summaries and manifest to:',output_dir)
print('Complete. No new inferential tests performed.')

Exported summaries and manifest to: /Users/ananyosen/Documents/Term Project/analysis/phase3_figures/figure4_log10_absolute_power
Complete. No new inferential tests performed.
